In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from diffusers import DDIMScheduler, DDPMPipeline
from matplotlib import pyplot as plt
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm
import os

# Loading & Sampling

In [ ]:
device = (
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

In [ ]:
image_pipe = DDPMPipeline.from_pretrained("google/ddpm-celebahq-256")

In [ ]:
image_pipe.to(device)

### Direct sampling

In [ ]:
images = image_pipe().images

In [ ]:
images[0]

### Faster sampling with Scheduler

In [ ]:
scheduler = DDIMScheduler.from_pretrained("google/ddpm-celebahq-256")

In [ ]:
def generate_image(pipeline, scheduler, timsteps=50):
    scheduler.set_timesteps(timsteps) # set the number of timestamps
    x = torch.randn(1, 3, 256, 256).to(device) # generate noise which will be transformed

    for i, t in tqdm(enumerate(scheduler.timesteps)):

        model_input = scheduler.scale_model_input(x, t) # get format

        with torch.no_grad():
            pred_noise = pipeline.unet(model_input, t)["sample"] # predict noise at current t

        scheduler_output = scheduler.step(pred_noise, t, x)
        x = scheduler_output.prev_sample  # update the smaple so far

        if i % 10 == 0 or i == len(scheduler.timesteps) - 1:
            fig, axs = plt.subplots(1, 2, figsize=(12, 5))

            grid = torchvision.utils.make_grid(x, nrow=4).permute(1, 2, 0)
            axs[0].imshow(grid.cpu().clip(-1, 1) * 0.5 + 0.5)
            axs[0].set_title(f"Current x (step {i})")

            pred_x0 = (
                scheduler_output.pred_original_sample
            )  # Not available for all schedulers
            grid = torchvision.utils.make_grid(pred_x0, nrow=4).permute(1, 2, 0)
            axs[1].imshow(grid.cpu().clip(-1, 1) * 0.5 + 0.5)
            axs[1].set_title(f"Predicted denoised images (step {i})")
            plt.show()

    return x


In [ ]:
image = generate_image(image_pipe, scheduler, 50)

# Finetuning (On Cats dataset)

## Loading the data  
Raw images => Tensor => Dataset => Dataloader

In [ ]:
# load image -> apply transformations -> dataset of all images
class ImageProcessor:

    def __init__(self, root_dir_path, transformations=None):
        self.root_dir_path  = root_dir_path
        self.transformations = transformations

        self.all_img_paths = [os.path.join(root_dir_path, img_path) for img_path in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self, key):
        img_path = self.all_img_paths[key]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img

In [ ]:
root_dir_path = "dataset/cats_set/"
transformations = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

In [ ]:
cats_dataset = ImageProcessor(root_dir_path, transformations)

In [ ]:
len(cats_dataset)

In [ ]:
data_loader = DataLoader(
    cats_dataset,
    batch_size=4,
    shuffle=True
)

## Load the pre trained Model & Scheduler

In [ ]:
image_pipe = DDPMPipeline.from_pretrained("google/ddpm-celebahq-256")
scheduler = DDIMScheduler.from_pretrained("google/ddpm-celebahq-256")

In [ ]:
if torch.backends.mps.is_available():
    device="mps"
elif torch.cuda.is_available():
    device="cuda"
else:
    device="cpu"
device = torch.device(device)

print(f"Device: {device}")

In [ ]:
image_pipe = image_pipe.to(device)

## Finetuning

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(
    image_pipe.unet.parameters(),
    lr=1e-5
)

In [ ]:
def fine_tune(
            data_loader: DataLoader, 
            image_pipe: DDPMPipeline, 
            scheudler: DDIMScheduler, 
            criterion: nn.modules.loss._Loss, 
            optimizer: optim.Optimizer, 
            num_epochs: int = 30
            ):

    for epoch in range(num_epochs):
        for steps, (images, *_) in tqdm(enumerate(data_loader)):
            images = images.to(device)
            images = images.unsqueeze(0)
            noise = torch.randn(images.shape).to(device) # create noise
            timesteps = torch.randint(
                0,
                image_pipe.scheduler.num_train_timesteps,
                (images.shape[0], ), 
                device=device,
            ).long() # select random timestamp
            noisy_images = image_pipe.scheduler.add_noise(
                images,
                noise,
                timesteps
            ) # add noise to images
            pred_noise = image_pipe.unet(
                noisy_images,
                timesteps,
                return_dict=False,
            )[0] # predict the noise

            loss = criterion(pred_noise, noise)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        show_images(epoch, image_pipe, scheduler)

def show_images(epoch, image_pipe: DDPMPipeline, scheduler: DDIMScheduler):
    x = torch.randn(8, 3, 256, 256).to(device)  # Batch of 8
    for i, t in tqdm(enumerate(scheduler.timesteps)):
        model_input = scheduler.scale_model_input(x, t)
        with torch.no_grad():
            noise_pred = image_pipe.unet(model_input, t)["sample"]
        x = scheduler.step(noise_pred, t, x).prev_sample

    grid = torchvision.utils.make_grid(x, nrow=4)
    plt.imshow(grid.permute(1, 2, 0).cpu().clip(-1, 1) * 0.5 + 0.5)
    plt.title(f"Epoch: {epoch}")

        

In [ ]:
# fine_tune(
#     data_loader,
#     image_pipe,
#     scheduler,
#     criterion,
#     optimizer
# )
# on Google collab

## Load the saved model

In [ ]:
image_pipe = DDPMPipeline.from_pretrained("saved_model")

In [ ]:
generate_image(image_pipe, scheduler, 500)